In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer # 변경된 부분
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 3가지 주제 설정 및 데이터 로드 (각 주제별 20개)
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
newsgroups = fetch_20newsgroups(subset='train',
                                categories=categories,
                                remove=('headers', 'footers', 'quotes'))

data = []
labels = []
target_names = newsgroups.target_names

for i, category in enumerate(categories):
    # 각 카테고리별로 처음 100개의 데이터만 추출
    cat_idx = np.where(newsgroups.target == i)[0][:500]
    for idx in cat_idx:
        data.append(newsgroups.data[idx])
        labels.append(target_names[i])

# 2. CountVectorizer로 변경
# 단순 빈도수를 계산하며, 의미 없는 단어 제거를 위해 stop_words를 설정합니다.
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

count_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 93635 stored elements and shape (1377, 21986)>

In [ ]:
import pandas as pd

# count_matrix를 DataFrame으로 변환
# Feature names (단어)를 컬럼명으로 사용합니다.
count_df = pd.DataFrame(count_matrix.toarray(), columns=vectorizer.get_feature_names_out())

count_df.head()

,00,000,0000,00000,000000,000005102000,000062david42,0001,000100255pixel,00041032,...,zurbrin,zurich,zurvanism,zvi,zwaartepunten,zwak,zwakke,zware,zwarte,zyxel
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
input_vector = vectorizer.transform([test_sentences[0]])
input_vector

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4 stored elements and shape (1, 21986)>

In [ ]:
similarities = cosine_similarity(input_vector, count_matrix)
similarities.shape

(1, 1377)

In [ ]:
np.argmax(similarities)

np.int64(818)

In [ ]:
best_idx = np.argmax(similarities)
best_score = similarities[0][best_idx]
best_score

np.float64(0.16666666666666666)

In [ ]:

# 3. 분류 함수
def classify_by_count(input_text):
    # 입력 문장을 빈도수 벡터로 변환
    input_vector = vectorizer.transform([input_text])

    # 코사인 유사도 계산
    similarities = cosine_similarity(input_vector, count_matrix)

    # 가장 높은 유사도 확인
    best_idx = np.argmax(similarities)
    best_score = similarities[0][best_idx]

    return labels[best_idx], best_score

# 4. 결과 확인
test_sentences = [
    "Exploring the mars with a robotic rover.",
    "The image resolution is too low for rendering.",
    "Arguments about the existence of God."
]

print(f"{'Input Sentence':<45} | {'Predicted Category':<20} | {'Similarity'}")
print("-" * 85)

for sentence in test_sentences:
    category, score = classify_by_count(sentence)
    print(f"{sentence[:43]:<45} | {category:<20} | {score:.4f}")

Input Sentence                                | Predicted Category   | Similarity
-------------------------------------------------------------------------------------
Exploring the mars with a robotic rover.      | sci.space            | 0.1667
The image resolution is too low for renderi   | comp.graphics        | 0.3213
Arguments about the existence of God.         | talk.religion.misc   | 0.4361
